# 01 - Data Preparation

Download Stockfish training data (.binpack), convert to flat bulletformat records
using a Rust CLI tool, and verify the data pipeline.

**Pipeline** (inspired by Bullet's approach):
1. Download `.binpack` from [Stockfish training data repos](https://robotmoon.com/nnue-training-data/)
2. Compile `binpack-to-bullet` (Rust, uses `sfbinpack` crate) to decompress into flat 32-byte records
3. Split into train/val/test `.data` files on Google Drive
4. Training notebooks memory-map these files and extract features on-the-fly

**Output**: `kanue/data/{train,val,test}.data` on Google Drive.

---

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/kanue'
!mkdir -p {DRIVE_BASE}/data {DRIVE_BASE}/checkpoints {DRIVE_BASE}/logs {DRIVE_BASE}/results

import os
from pathlib import Path
DATA_DIR = Path(DRIVE_BASE) / 'data'

## 1. Install Rust & Compile binpack-to-bullet

Small Rust CLI using the `sfbinpack` crate (same one Bullet uses)
to decompress `.binpack` into flat 32-byte bulletformat records.
Compiles in ~1-2 minutes.

In [ ]:
# Install Rust if not present
if not os.path.exists('/root/.cargo/bin/cargo'):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']
!rustc --version

In [ ]:
# Clone kanue repo and build the converter
if not os.path.exists('/content/kanue'):
    !git clone --depth 1 https://github.com/y0sif/kanue.git /content/kanue

CONVERTER = '/content/kanue/tools/binpack-to-bullet/target/release/binpack-to-bullet'
if not os.path.exists(CONVERTER):
    !cd /content/kanue/tools/binpack-to-bullet && cargo build --release 2>&1 | tail -5
else:
    print('Converter already built')

!ls -lh {CONVERTER}

## 2. Download Training Data

We download a Stockfish `.binpack` file from HuggingFace (public, no auth needed).

Options (pick one in the cell below):
- **test77-jan2022** (~1.3 GB compressed) -- good for initial experiments
- **fishpack32** (~5.5 GB uncompressed) -- official Stockfish, no decompression needed
- [More datasets](https://robotmoon.com/nnue-training-data/)

In [ ]:
RAW_DIR = Path('/content/raw_data')
RAW_DIR.mkdir(exist_ok=True)

BINPACK_PATH = RAW_DIR / 'training.binpack'

if not BINPACK_PATH.exists():
    # Option A: test77 from HuggingFace (~1.3 GB compressed, needs zstd)
    !apt-get install -y -qq zstd
    !wget -q --show-progress "https://huggingface.co/datasets/linrock/test77/resolve/main/test77-2022-01-jan-2tb7p.binpack.zst" -O {RAW_DIR}/training.binpack.zst
    !zstd -d {RAW_DIR}/training.binpack.zst -o {BINPACK_PATH} --rm

    # Option B (alternative): fishpack32 from official Stockfish (~5.5 GB, no decompression)
    # !wget -q --show-progress "https://huggingface.co/datasets/official-stockfish/master-binpacks/resolve/main/fishpack32.binpack" -O {BINPACK_PATH}
else:
    print(f'Binpack already downloaded: {BINPACK_PATH}')

!ls -lh {BINPACK_PATH}

## 3. Convert binpack to bulletformat

Decompresses the binpack into flat 32-byte records.
Filters out positions in check and extreme evaluations.
This is what Bullet does on-the-fly -- we do it once and save.

In [ ]:
BULLET_PATH = RAW_DIR / 'all_positions.data'

if not BULLET_PATH.exists():
    # Convert -- adjust --max-positions for initial experiments (remove for full dataset)
    !{CONVERTER} {BINPACK_PATH} {BULLET_PATH} --max-positions 10000000 --max-score 3000
else:
    print(f'Already converted: {BULLET_PATH}')

file_size = BULLET_PATH.stat().st_size
n_positions = file_size // 32
print(f'\n{n_positions:,} positions ({file_size / 1024 / 1024:.1f} MB)')

## 4. Shuffle and Split

Shuffle the flat binary records (crucial for training quality)
and split into train/val/test on Google Drive.

In [ ]:
import numpy as np

RECORD_SIZE = 32

# Check if already split
if (DATA_DIR / 'train.data').exists():
    train_size = (DATA_DIR / 'train.data').stat().st_size // RECORD_SIZE
    val_size = (DATA_DIR / 'val.data').stat().st_size // RECORD_SIZE
    test_size = (DATA_DIR / 'test.data').stat().st_size // RECORD_SIZE
    print(f'Data already split on Drive:')
    print(f'  Train: {train_size:,} | Val: {val_size:,} | Test: {test_size:,}')
    print('Delete the .data files in Drive and re-run to regenerate.')
else:
    # Memory-map the full file
    data = np.memmap(str(BULLET_PATH), dtype=np.uint8, mode='r')
    n = len(data) // RECORD_SIZE
    data = data[:n * RECORD_SIZE].reshape(n, RECORD_SIZE)

    # Shuffle indices
    print(f'Shuffling {n:,} positions...')
    indices = np.random.permutation(n)

    # 80/10/10 split
    train_end = int(0.8 * n)
    val_end = int(0.9 * n)

    splits = {
        'train': indices[:train_end],
        'val': indices[train_end:val_end],
        'test': indices[val_end:],
    }

    for name, idx in splits.items():
        path = DATA_DIR / f'{name}.data'
        out = np.memmap(str(path), dtype=np.uint8, mode='w+',
                        shape=(len(idx), RECORD_SIZE))
        # Write in chunks to avoid memory issues
        CHUNK = 500_000
        for start in range(0, len(idx), CHUNK):
            end = min(start + CHUNK, len(idx))
            out[start:end] = data[idx[start:end]]
        out.flush()
        del out
        size_mb = path.stat().st_size / 1024 / 1024
        print(f'  {name}: {len(idx):,} positions ({size_mb:.1f} MB)')

    del data
    print('\nDone. Data saved to Google Drive.')

## 5. Verify Data Pipeline

Test that the bulletformat loader reads data and extracts features correctly.

In [ ]:
import sys
sys.path.insert(0, '/content/kanue/src')
!pip install -q python-chess tqdm

In [ ]:
from kanue.data import BulletformatBatchDataset, load_bulletformat
import torch

# Load and inspect raw records
train_records = load_bulletformat(DATA_DIR / 'train.data')
print(f'Train records: {len(train_records):,}')
print(f'Score range: [{train_records["score"].min()}, {train_records["score"].max()}]')
print(f'Result distribution: loss={np.sum(train_records["result"]==0):,}, '
      f'draw={np.sum(train_records["result"]==1):,}, '
      f'win={np.sum(train_records["result"]==2):,}')

# Test the batch dataset
ds = BulletformatBatchDataset(DATA_DIR / 'train.data', batch_size=16384, shuffle=True)
batch = next(iter(ds))
stm, nstm, targets = batch

print(f'\nBatch shapes: stm={stm.shape}, nstm={nstm.shape}, targets={targets.shape}')
print(f'STM active features per position: {(stm >= 0).sum(dim=1).float().mean():.1f}')
print(f'Target range: [{targets.min():.4f}, {targets.max():.4f}]')
print(f'Target mean: {targets.mean():.4f}')

In [ ]:
# Smoke test: models accept sparse format
from kanue.models import NnBoard768, KanBoard768

s, n, t = stm[:4], nstm[:4], targets[:4]

baseline = NnBoard768(32)
out = baseline(s, n)
print(f'Baseline output: {out.shape}, values: {out.detach().flatten().tolist()[:4]}')

kan = KanBoard768(32, grid_size=3)
out = kan(s, n)
print(f'KAN output: {out.shape}, values: {out.detach().flatten().tolist()[:4]}')

print('\nData pipeline verified. Proceed to notebook 02.')

## 6. Data Distribution (optional)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Score distribution (sample for speed)
scores = train_records['score'][:1_000_000]
axes[0].hist(scores, bins=100, alpha=0.7)
axes[0].set_xlabel('Score (centipawns)')
axes[0].set_ylabel('Count')
axes[0].set_title('Score Distribution')

# Target distribution
targets_sample = 1.0 / (1.0 + np.exp(-scores.astype(np.float32) / 400.0))
axes[1].hist(targets_sample, bins=50, alpha=0.7)
axes[1].set_xlabel('Target (win probability)')
axes[1].set_ylabel('Count')
axes[1].set_title('Target Distribution')

# Result distribution
results = train_records['result'][:1_000_000]
labels = ['Loss (0)', 'Draw (1)', 'Win (2)']
counts = [np.sum(results == i) for i in range(3)]
axes[2].bar(labels, counts, alpha=0.7)
axes[2].set_ylabel('Count')
axes[2].set_title('Game Result Distribution')

plt.tight_layout()
plt.savefig(str(DATA_DIR.parent / 'results' / 'data_distribution.png'), dpi=150)
plt.show()